# Inflation of a simplified left ventricle

**Joakim Sundnes**

Date: **June 27, 2023**


## Step 1: Run the mechanics solver on an ellipsoid
The first part of this notebook includes complete code for a simulation of a very simplified left ventricle. The geometry is a truncated and thick-walled ellipsoid, and we only model the filling phase of the cardiac cycle. Most of the code is very similar to the previous example using a unit cube, but some additional functions have been added to handle the more complex geometry and fiber architecture. Step through the code cells and familiarize yourself with how the model works and what it outputs.


### Model outline
We will now increase the realism slightly, and consider the classic ellipsoid approximation of the left ventricle (LV). We will use the Guccione material model introduced previously, and the additional complexity is mainly in the geometry and that fiber orientation is not constant.

We will look at two simple cases:
* Passive inflation: the ellipsoidal LV is simply inflated, by ramping up the endocardial pressure.
* Active contraction: the LV contracts against a constant endocardial pressure.

As before, the mathematical problem to be solved is a quasi-static hyper-elasticity problem, with a combination of Dirichlet and Neumann boundary conditions:

\begin{align*}
- \nabla\cdot P &= 0 && \quad \text{ in } \Omega \\
  u &= 0 && \quad \text{ on } \Gamma_{\mathrm{D}} \\
  P \cdot n &= T && \quad \text{ on } \Gamma_{\mathrm{N}} \\
  P \cdot n &= 0 && \quad \text{ on } \Gamma_{\mathrm{0}}
\end{align*}

Again, $P$ is the first Piola-Kirchhoff stress tensor, $u$ is the displacement, $T$ is a load applied to parts of the boundary, and $\Omega, \Gamma_{\mathrm{D}}, \Gamma_{\mathrm{N}},\Gamma_{\mathrm{0}}$ are the domain and the boundaries for Dirichlet- and Neumann boundary conditions, respectively.

In [ ]:
from dolfinx import fem, io, plot, default_scalar_type
import dolfinx.fem.petsc
from ufl import (
    TestFunction,
    Measure,
    FacetNormal,
    variable,
    Identity,
    SpatialCoordinate,
    grad,
    diff,
    dot,
    inner,
    tr,
    det,
    inv,
    dx,
    as_vector,
)
from mpi4py import MPI
import io4dolfinx

import pyvista
import matplotlib.pyplot as plt
import numpy as np

from guccionematerial import GuccioneMaterial
from plotting import setup_gif_visualizer, update_gif_frame, plot_fibers

The domain is a truncated ellipsoid generated with [cardiac-geometriesx](https://github.com/ComputationalPhysiology/cardiac-geometriesx/tree/main) and is shown here:

In [ ]:
with io.XDMFFile(MPI.COMM_WORLD, "lv-mesh/mesh.xdmf", "r") as xdmf:
    domain = xdmf.read_mesh(name="Mesh")

# Plot the mesh with Pyvista
pyvista.set_jupyter_backend("static")
pv_grid = pyvista.UnstructuredGrid(*plot.vtk_mesh(domain))
plotter = pyvista.Plotter(window_size=[300, 300])
plotter.add_mesh(pv_grid, show_edges=True)
plotter.show_axes()
plotter.show()

The mesh is very coarse to minimize the compute time, and therefore looks edgy and non-smooth. For simplicity, we will hold the base of the LV fixed in all directions, the epicardial surface is unloaded, while a time-varying pressure is applied to the endocardial surface.

As above, we derive the weak form to apply the finite element method. Multiplying with a test function $v \in \hat{V}$, integrating by parts and applying the boundary conditions leads to the problem:

Find $u \in V$ such that

$$
\begin{equation*}
    \int_{\Omega} P : \nabla v dx
    = \int_{\Gamma_{\mathrm{N}}} T \cdot v ds
  \end{equation*}
$$

We are now ready to set up our FEniCSx solver. 

In addition to the standard imports and settings, it will be useful to define a couple of helper functions to handle the pre- and post-processing. The function `load_ellipsoid_data` reads the geometry and fiber information from a set of files, and returns a FEniCSx mesh and a list of functions defining the boundary markers and fiber orientation:

In [ ]:
def load_ellipsoid_data():
    """Returns 4-tuple:
    domain - the mesh,
    mf - MeshTags defining boundary markers,
    numbering - dict of marking numbers,
    fibers - list of functions defining microstructure
    """

    # Load the mesh and boundary markers (MeshTags) from XDMF
    with io.XDMFFile(MPI.COMM_WORLD, "lv-mesh/mesh.xdmf", "r") as xdmf:
        # Check your XDMF file to ensure the name="Mesh" matches
        domain = xdmf.read_mesh(name="Mesh")

        # In FEniCSx, we must compute connectivity before dealing with facets
        domain.topology.create_connectivity(
            domain.topology.dim - 1, domain.topology.dim
        )

        # Read the facet tags
        mf = xdmf.read_meshtags(domain, name="Facet tags")

    # Scale mesh from millimeters to centimeters
    domain.geometry.x[:] *= 0.1

    numbering = {"BASE": 5, "ENDO": 6, "EPI": 7}

    # Setup function space and functions for fibers
    fiber_space = fem.functionspace(domain, ("Lagrange", 2, (domain.geometry.dim,)))
    fiber = fem.Function(fiber_space, name="f0")
    sheet = fem.Function(fiber_space, name="s0")
    cross_sheet = fem.Function(fiber_space, name="n0")

    # Read the fiber directions from file using io4dolfinx
    bp_filepath = "lv-mesh/geometry.bp"
    io4dolfinx.read_function(bp_filepath, fiber)
    io4dolfinx.read_function(bp_filepath, sheet)
    io4dolfinx.read_function(bp_filepath, cross_sheet)

    fibers = [fiber, sheet, cross_sheet]

    return domain, mf, numbering, fibers

We will use a transversely isotropic material model, so the only direction that really matters is the fiber orientation, but the solver requires the sheet and sheet normal directions to be defined. The fiber field stored in the file `lv-mesh/geometry.bp` is a standard approximation where the fiber angle varies linearly from -60 to +60 degrees going from the endo- to epicardium, as illustrated in the figure below.

In [ ]:
domain, facet_tags, numbering, fibers = load_ellipsoid_data()

f0, s0, n0 = fibers

plot_fibers(
    domain,
    fiber_funcs=[f0, s0, n0],
    names=["Fiber (f0)", "Sheet (s0)", "Cross-Sheet (n0)"],
    colors=["red", "green", "blue"],
    arrow_scale=0.25,
)


When doing mechanics simulations of the entire LV, it is interesting to look at the variations in cavity volume, to plot PV loops etc. For computing the cavity volume, we apply a small mathematical trick to turn a volume integral into a surface integral:

$$
V = \int_{cavity} 1 dx = -1/3\int_{\Gamma_N} x\cdot n ds ,
$$

where $x$ is the coordinate of the *deformed* surface, and $n$ is the deformed surface normal. With a Lagrangian approach we map everything to the reference configuration, and define the problem in terms of the displacement $u$, to get the following code:

In [ ]:
def compute_cavity_volume(mesh, mf, numbering, u=None):
    X = SpatialCoordinate(mesh)
    N = FacetNormal(mesh)

    if u is not None:
        I = Identity(3)
        F = I + grad(u)
        J = det(F)
        vol_form = (-1.0 / 3.0) * dot(X + u, J * inv(F).T * N)
    else:
        vol_form = (-1.0 / 3.0) * dot(X, N)

    ds = Measure("ds", domain=mesh, subdomain_data=mf)

    return fem.assemble_scalar(fem.form(vol_form * ds(numbering["ENDO"])))

The input to the function is a FEniCSx mesh, a set of MeshTags defining the boundary markers, a dictionary containing the names of the boundary markers, and an optional displacement $u$. The function assumes that the endocardial surface is marked with boundary marker "ENDO". 

Having these two functions in place, the rest of the code is very similar to the unit cube problems considered previously. In fact, most of the additional complexity of working with an ellipsoid and non-constant fiber fields is handled in the 'load_ellipsoid_geometry' function. Once this function has been called to set up the boundary markers and fiber fields, the rest of the problem definition looks very familiar. The complete code is contained in the two cells below.

### Define the geometry, boundary conditions, and the weak form:

In [ ]:
domain, facet_tags, numbering, fibers = load_ellipsoid_data()

V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))

# Redefine boundary measure, to allow integration over parts of boundary
metadata = {"quadrature_degree": 4}
ds = Measure("ds", domain=domain, subdomain_data=facet_tags, metadata=metadata)
dx = Measure("dx", domain=domain, metadata=metadata)

base_dofs = fem.locate_dofs_topological(
    V, facet_tags.dim, facet_tags.find(numbering["BASE"])
)

bc = fem.dirichletbc(np.zeros(3, dtype=default_scalar_type), base_dofs, V)
bcs = [bc]

# Define solution u and test function v
u = fem.Function(V)
v = TestFunction(V)

# Define strain measures
I = Identity(3)  # the identity matrix
F = I + grad(u)  # the deformation gradient
F = variable(F)

mat = GuccioneMaterial(
    domain, e1=fibers[0], e2=fibers[1], e3=fibers[2], kappa=1e3, Tactive=0.0
)

psi = mat.strain_energy(F)
P = diff(psi, F)  # the first Piola-Kirchoff stress tensor

p_endo = fem.Constant(domain, 0.0)

# Define nonlinear problem
N = FacetNormal(domain)
Gext = (
    p_endo * inner(v, det(F) * inv(F) * N) * ds(numbering["ENDO"])
)  # endocardial pressure
R = inner(P, grad(v)) * dx - Gext

### Then solve the problem with a gradually increasing pressure

Run the code cell below. You will see a partial pressure-volume curve, which only contains the filling phase. The full displacement field is saved for each load step, and can be visualized in Paraview. 

In [ ]:
# Step-wise loading
pressure_steps = 20
target_pressure = 10.0

# Loop over load steps:
pressures = np.linspace(0, target_pressure, pressure_steps)
volumes = np.zeros_like(pressures)

filename = "output/ellipsoid_inflation.bp"
outfile = io.VTXWriter(domain.comm, filename, [u])
outfile.write(0.0)

petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
 #   "snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="ellipsoid_inflation_",
)

plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/ellipsoid_inflation.gif", clim=[0, 0.4]
)

for step in range(pressure_steps):
    p_endo.value = -pressures[step]
    problem.solve()
    # Compute and store volume for each step:
    volumes[step] = compute_cavity_volume(domain, facet_tags, numbering, u)

    outfile.write(pressures[step])
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

outfile.close()
plotter.close()

plt.plot(volumes, pressures)
plt.xlabel("Volume")
plt.ylabel("Pressure")
plt.show()

In [ ]:
# Display the generated GIF
from IPython.display import Image

Image(filename="output/ellipsoid_inflation.gif", width=500)

## Step 2: Add active contraction
The final step is to add active contraction in our ellipsoid problem. As we saw in the unit cube examples, the `GuccioneMaterial` class includes functionality for active stress. While simply ramping up the pressure worked is a reasonably realistic approximation of the filling phase of the cardiac cycle, proper simulation of the complete cardiac cycle requires far more complex boundary conditions. For a full cardiac cycle, the pressure volume relation (PV loop) looks as follows:
<img src="figs/pv_loop.png" width=400>

This loop has four characteristic phases, with different hemodynamic boundary conditions:
* The filling phase, where pressure and volume both increase. This is the phase we modeled above, by simply ramping up the endocardial pressure. In a real LV, the mitral valve is open during this phase, while the aortic valve is closed. 
* As the ventricle starts contracting, the LV pressure becomes higher than the pressure in the pulmonary veins, which causes the mitral valve to close. This marks the end of diastole (ED). The heart enters the *isovolumic contraction* phase, where the pressure increases while the volume stays constant since both valves are closed. In a heart mechanics model, realistic boundary conditions would require solving a non-linear equation to find the cavity pressure that keeps the volume constant as active stress increases.
* At some point the LV pressure will exceed the aortic pressure, and the aortic valve opens and allows blood to be ejected into the systemic circulation. This is the *ejection phase*, and the endocardial pressure will be determined by the contractile force and the compliance and resistance of the systemic circulation. The typical boundary condition to use for this phase is a so-called Windkessel model.
* Finally, as the pressure in the LV starts to drop, the aortic valve closes and the heart enters the *isovolumic relaxation* phase. This is the end-systolic (ES) point. In terms of boundary conditions this phase is identical to isovolumic contraction.

One can define relatively simple models that give a reasonably realistic representation of the four phases above. However, in this course we will keep things even simpler, and simply let our LV contract against a constant endocardial pressure. The resulting simulation will be a reasonably realistic filling phase, followed by a less realistic ejection phase (and no isovolumic phase in between). The simulation protocol should look something like this:
* First keep the tissue passive, and inflate the LV to a given pressure (filling phase)
* Then, hold the pressure constant while ramping up the active stress

In code, the time course of endocardial pressure and active stress will look something like:

In [ ]:
# Step-wise loading
pressure_steps = 20
active_steps = 20
target_pressure = 10.0
target_active = 40.0

# first ramp up pressure, then keep constant
filling_pressure = np.linspace(0, target_pressure, pressure_steps)
const_pressure = np.ones(active_steps) * target_pressure
pressures = np.concatenate((filling_pressure, const_pressure))

# zero active tension during filling, then increase linearly
active1 = np.zeros_like(filling_pressure)
active2 = np.linspace(0, target_active, active_steps)
active = np.concatenate((active1, active2))

plt.plot(active)
plt.plot(pressures)
plt.legend(["Active stress", "LV pressure"])
plt.xlabel("Time")
plt.ylabel("Pressure/active stress")
plt.axis([0, 40, -1, 41])

plt.show()

## Final exercise

```{exercise} Solve the ellipsoid inflation-contraction problem
:label: l13-ellipsoid

* Add the varying pressure- and active stress defined above to the LV solver. 
* What does the PV loop look like in this case? Is it as expected?
* Optional: How can you change the solver to simulate the isovolumic contraction phase?
```

In [ ]:
domain, facet_tags, numbering, fibers = load_ellipsoid_data()

V = fem.functionspace(domain, ("Lagrange", 1, (domain.geometry.dim,)))

# Redefine boundary measure, to allow integration over parts of boundary
metadata = {"quadrature_degree": 4}
ds = Measure("ds", domain=domain, subdomain_data=facet_tags, metadata=metadata)
dx = Measure("dx", domain=domain, metadata=metadata)

clamp = np.zeros(domain.geometry.dim, dtype=default_scalar_type)
base_dofs = fem.locate_dofs_topological(
    V, facet_tags.dim, facet_tags.find(numbering["BASE"])
)
bc = fem.dirichletbc(clamp, base_dofs, V)
bcs = [bc]

# Define solution u and test function v
u = fem.Function(V)
v = TestFunction(V)

# Define strain measures
I = Identity(3)  # the identity matrix
F = I + grad(u)  # the deformation gradient
F = variable(F)

mat = GuccioneMaterial(
    domain, e1=fibers[0], e2=fibers[1], e3=fibers[2], kappa=1e3, Tactive=0.0
)
psi = mat.strain_energy(F)
P = diff(psi, F)  # the first Piola-Kirchoff stress tensor

p_endo = fem.Constant(domain, 0.0)

# Define nonlinear problem
N = FacetNormal(domain)
Gext = (
    p_endo * inner(v, det(F) * inv(F) * N) * ds(numbering["ENDO"])
)  # endocardial pressure
R = inner(P, grad(v)) * dx - Gext

# Step-wise loading
#
# EXERCISE: Add code here
#


petsc_options = {
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
    #"snes_monitor": None,
}
problem = fem.petsc.NonlinearProblem(
    R,
    u,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="ellipsoid_contraction_",
)

plotter, grid, magnitude, us_expr, actor = setup_gif_visualizer(
    domain, u, filename="output/ellipsoid_contraction.gif", clim=[0, 0.4]
)

filename = "output/ellipsoid_contraction.bp"
outfile = io.VTXWriter(domain.comm, filename, [u])
outfile.write(0.0)


volumes = np.zeros_like(pressures)
for i, step in enumerate(range(pressure_steps + active_steps)):
    # Assign pressure and active stress
    # 
    # EXERCISE: Add code here
    # 

    problem.solve()
    volumes[step] = compute_cavity_volume(domain, facet_tags, numbering, u)
    
    outfile.write(i)
    update_gif_frame(plotter, grid, u, magnitude, us_expr, actor)

plotter.close()
outfile.close()

In [ ]:
# Display the generated GIF
from IPython.display import Image
Image(filename="output/ellipsoid_contraction.gif", width=500)

In [ ]:
plt.figure()
plt.plot(volumes, pressures)
plt.xlabel("Volume")
plt.ylabel("Pressure")
plt.show()